In [70]:
import torch
from einops import rearrange, einsum

In [33]:
images = torch.randn(64, 128, 128, 3)
dim_by = torch.linspace(start=0.0, end=1.0, steps=10)

In [34]:
dim_value = rearrange(dim_by, 'dim_value -> 1 dim_value 1 1 1')

In [35]:
dim_value.shape

torch.Size([1, 10, 1, 1, 1])

In [36]:
images_rearr = rearrange(images, 'b height width channel -> b 1 height width channel')

In [37]:
images_rearr.shape

torch.Size([64, 1, 128, 128, 3])

In [38]:
dimmed_images = images_rearr * dim_value

In [39]:
dimmed_images.shape

torch.Size([64, 10, 128, 128, 3])

In [41]:
dimmed_images = einsum(
    images, dim_by,
    "batch height width channel, dim_value -> batch dim_value height width channel"
)

In [54]:
A = torch.randn((3,5))
B = torch.randn((5,4))

In [58]:
A@B

tensor([[-3.5395,  2.3751, -3.2585,  0.6191],
        [-4.1558, -0.8302, -2.6138, -6.3266],
        [-0.3607,  1.3765, -0.1282, -0.0657]])

In [60]:
einsum(A, A, "i j, i j -> i j")

tensor([[0.1227, 0.0632, 0.4799, 5.6157, 0.6954],
        [0.2592, 2.6478, 0.1485, 2.1493, 0.2124],
        [0.3179, 0.0307, 1.7696, 0.2864, 0.0949]])

In [68]:
channels_last = torch.randn(64, 32, 32, 3)
B = torch.randn(32*32, 32*32)

In [104]:
A.shape

torch.Size([32, 32])

In [ ]:
C = B @ rearrange(A, "row col -> (row col)")
C

torch.Size([1024])

In [121]:
A = torch.randn(64, 32, 32, 3)

In [122]:
B = torch.randn(32, 32)

In [127]:
torch.all(einsum(B, A, "i j, b j k c -> b i k c") == B @ A)

tensor(False)

In [162]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from einops import einsum

class Linear(nn.Module):
    def __init__(self, in_features, out_features, device=None, dtype=None):
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.W = nn.Parameter(torch.empty((out_features, in_features), device=device, dtype=dtype))
        nn.init.trunc_normal_(self.W, std=(2 / (in_features + out_features)) ** 0.5)

    def forward(self, x: torch.Tensor):
        return einsum(self.W, x, "i j, ... j -> ... i")

In [156]:
model = Linear(4, 32, device=None, dtype=float)

In [159]:
x = torch.randn(4, dtype=float)

In [3]:
import torch
A = torch.randn(10,20,2)

In [5]:
def f(x):
    return x**2

In [10]:
A.shape

torch.Size([10, 20, 2])

In [ ]:
from einops import einsum, repeat
einsum(A, A, "... i, ... i -> ...").shape

torch.Size([10, 20])

In [41]:
eps = 1e-2
batch_size = 10
sequence_length = 5

In [45]:
repeat(torch.tensor([eps]), "eps -> (eps 3) 5")

tensor([[0.0100, 0.0100, 0.0100, 0.0100, 0.0100],
        [0.0100, 0.0100, 0.0100, 0.0100, 0.0100],
        [0.0100, 0.0100, 0.0100, 0.0100, 0.0100]])

In [58]:
A = torch.randn(1, 4)
B = torch.tensor([1,2,3])

In [59]:
A

tensor([[ 0.2160, -0.1648, -1.0402,  2.6934]])

In [61]:
einsum(A, B, "batch seq, dim -> batch seq dim").shape

torch.Size([1, 4, 3])

In [64]:
x = torch.linspace(1, 10, 10)

In [66]:
einsum(x,x, "... i, ... i -> i")

tensor([  1.,   4.,   9.,  16.,  25.,  36.,  49.,  64.,  81., 100.])

In [43]:
repeat(torch.tensor([1]), "i -> (i 10) 20")

tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])